# Fine-Tuning BERT for Aspect-Based Sentiment Analysis

This notebook demonstrates how to fine-tune a pre-trained transformer model (`bert-base-uncased`) to predict aspect-level sentiment. 

We provide the model with a sentence AND a specific extracted aspect. The model learns which sentiment applies to which aspect, allowing it to correctly classify mixed-sentiment sentences.

In [ ]:
import os
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## 1. Prepare Dataset
We need a dataset with columns: `review`, `aspect`, `sentiment` (where sentiment is 0=negative, 1=neutral, 2=positive).

In [ ]:
# Load or generate dataset
data_path = "../data/processed/absa_training.csv"
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print("Using synthetic dummy data for demonstration.")
    data = [
        {"review": "The actor was brilliant but the plot was boring.", "aspect": "actor", "sentiment": 2},
        {"review": "The actor was brilliant but the plot was boring.", "aspect": "plot", "sentiment": 0},
    ] * 100
    df = pd.DataFrame(data)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
train_df.head()

## 2. Tokenization
We use `bert-base-uncased`. Notice how we pass BOTH `review` and `aspect` to the tokenizer. BERT separates them with a `[SEP]` token.

In [ ]:
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, texts, aspects, labels, tokenizer, max_length=512):
        self.texts = texts
        self.aspects = aspects
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            text_pair=self.aspects[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length"
        )
        item = {key: torch.tensor(val) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ABSADataset(train_df['review'].tolist(), train_df['aspect'].tolist(), train_df['sentiment'].tolist(), tokenizer)
val_dataset = ABSADataset(val_df['review'].tolist(), val_df['aspect'].tolist(), val_df['sentiment'].tolist(), tokenizer)

## 3. Fine-Tuning
We train the model using HuggingFace's `Trainer` API.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir="../results/transformer_absa",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# trainer.train() # Uncomment to train

## 4. Save and Test
Save the model to `models/bert_absa` so it can be picked up by the Flask API.

In [ ]:
# model.save_pretrained("../models/bert_absa")
# tokenizer.save_pretrained("../models/bert_absa")
print("Model saved and ready for deployment.")